#FGS Exposure (run by hand)

In [ ]:
# =============================================================================
# master/Master_exposure.py
# Master notebook for the exposure chain. RUN BY HAND. Do not schedule it.
#
# What it runs, in order:
#   33  build_address_points        address points with people, households and the flood band at each
#   34  build_exposure_grid         the 100 m exposure grid, built from the address points
#   35  build_warning_area_exposure the people, households and critical sites inside each warning and alert area
#
# WHEN TO RUN IT
#   from_step = 35  The usual case. Run it after the flood areas change (Master_weekly refreshes them), so the area figures follow the new boundaries.
#   from_step = 34  After job 33 has been rebuilt on its own, or after the critical sites (job 31) change.
#   from_step = 33  A full rebuild: after the National Statistics UPRN Lookup, the Census output areas or the hazard rasters (job 30) change.
#                   Job 33 is long. It always runs with resume = yes: a tile is finished when its rows are in the table, so a re-run skips finished tiles.
#                   resume = no is not offered. It rebuilds every tile and has failed on serverless compute (start-up races on the tile tables, driver memory).
#
# WHAT IT DOES NOT RUN
#   29 census and deprivation, 30 hazard rasters, 31 critical sites, 32 responder geographies. They need a person to point them at a new source file or
#   page, or they change only when that source does. Run them by hand first if their source has changed, then this notebook from step 33.
#   38 economics assumptions is also by hand: its table is edited by people.
#
# IF A STEP FINISHES WITH FAILED TILES
#   The notebook stops. Run it again with the same from_step. Tiles that finished are skipped, so only the failed ones are retried.
#
# The timeouts are generous guesses, not measured. Adjust them once you know how long each step takes on your compute.
# =============================================================================

dbutils.widgets.dropdown("from_step", "35", ["33", "34", "35"], "Start from job")
FROM_STEP = int(dbutils.widgets.get("from_step"))

print(f"Exposure chain, starting from job {FROM_STEP}.")


def run_step(number, label, path, timeout_seconds, arguments):
    """Run one job. Stops the chain unless it reports plain success."""
    print("=" * 60)
    print(f"STEP {number}: {label}")
    print("=" * 60)
    result = dbutils.notebook.run(path=path, timeout_seconds=timeout_seconds, arguments=arguments)
    print(f"Job {number} result: {result}")
    if result != "success":
        raise Exception(f"Job {number} ({label}) ended with: {result}. Run this notebook again with from_step = {number}; finished tiles are skipped.")

In [ ]:
# =============================================================================
# STEP 33: ADDRESS POINTS
# The long step. resume is always yes, so a re-run picks up only the tiles that are not in the table.
# =============================================================================
if FROM_STEP <= 33:
    run_step(33, "Build address points", "../jobs/33_build_address_points", 14400, {"resume": "yes"})
else:
    print("Skipping job 33.")

In [ ]:
# =============================================================================
# STEP 34: EXPOSURE GRID
# Needs job 33 to have finished. Uses the critical sites if job 31 has built them.
# =============================================================================
if FROM_STEP <= 34:
    run_step(34, "Build exposure grid", "../jobs/34_build_exposure_grid", 7200, {})
else:
    print("Skipping job 34.")

In [ ]:
# =============================================================================
# STEP 35: WARNING AREA EXPOSURE
# Needs job 33 to have finished, and reads the flood areas as they stand now.
# =============================================================================
run_step(35, "Build warning area exposure", "../jobs/35_build_warning_area_exposure", 7200, {})

print("Exposure chain complete.")
dbutils.notebook.exit("success")